In [1]:
!pip install langchain langchain-community langchain-openai langchain-classic langchain-core wikipedia python-docx -q

In [2]:
# IMPORTS & DEPENDENCIES

from datetime import datetime
import docx
from pathlib import Path
import os
import math
import wikipedia

from langchain_openai import ChatOpenAI
from langchain_classic.agents import Tool, initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.tools import Tool as CoreTool

print("✓ All dependencies imported successfully")

✓ All dependencies imported successfully


In [3]:
# LLM CONNECTION - GEMMA 3 4B

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed",
    model="google/gemma-3-4b",
    temperature=0.2
)

print("✓ Connected to Gemma 3 4B model")
print("  API: http://127.0.0.1:1234/v1")
print("  Model: google/gemma-3-4b")
print("  Temperature: 0.2 (balanced mode)")

✓ Connected to Gemma 3 4B model
  API: http://127.0.0.1:1234/v1
  Model: google/gemma-3-4b
  Temperature: 0.2 (balanced mode)


In [4]:
# DEFINE TOOLS (3 Essential Tools)

# 1. ARITHMETIC CALCULATOR
def tool_calculator(query):
    """Solve arithmetic problems like addition, subtraction, multiplication, and division"""
    try:
        result = eval(query, {"__builtins__": None}, {"sqrt": math.sqrt, "pi": math.pi, "e": math.e})
        return str(result)
    except Exception as e:
        return f"Invalid arithmetic expression: {str(e)}"

# 2. WIKIPEDIA SEARCH
def tool_wikipedia(query):
    """Search Wikipedia and return a summary of a topic"""
    try:
        summary = wikipedia.summary(query, sentences=5)
        return summary
    except Exception as e:
        return f"Error: {str(e)}"

# 3. DOCUMENT CREATOR
def create_word_doc(content, filename="generated.docx", folder="generated_docs"):
    """Helper function to create Word documents"""
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    
    doc = docx.Document()
    doc.add_heading("Generated Document", 0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph("="*60)
    
    for line in content.split("\n"):
        if line.strip():
            doc.add_paragraph(line)
    
    doc.save(filepath)
    return f"Document saved at: {filepath}"

def word_tool(query):
    """Creates a Word document from user request"""
    try:
        prompt = f"""Create a well-structured document based on this request:

{query}

Format the output with:
- Title
- Clear sections
- Bullet points where appropriate
- Professional formatting"""
        
        content = llm.invoke(prompt).content
        filename = f"doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
        return create_word_doc(content, filename)
    except Exception as e:
        return f"Error creating document: {str(e)}"

# Create Tool objects
calculator_tool = Tool(
    name="Calculator",
    func=tool_calculator,
    description="Solve math problems. Use for: calculations, math expressions, arithmetic"
)

wikipedia_tool = Tool(
    name="Wikipedia Search",
    func=tool_wikipedia,
    description="Search Wikipedia for information. Use for: facts, information, knowledge about topics"
)

word_document_tool = Tool(
    name="Generate Word Document",
    func=word_tool,
    description="Create or generate a Word (.docx) document. Use for: create document, generate document, write document, make word doc, generate word"
)

print("✓ Tools configured:")
print("  1. Calculator - Math expressions")
print("  2. Wikipedia Search - Information lookup")
print("  3. Generate Word Document - Document creation")


✓ Tools configured:
  1. Calculator - Math expressions
  2. Wikipedia Search - Information lookup
  3. Generate Word Document - Document creation


In [5]:
# SETUP AGENT WITH MEMORY AND DECISION RULES

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="input",
    return_messages=True
)

tools = [calculator_tool, wikipedia_tool, word_document_tool]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    return_intermediate_steps=False,
    handle_parsing_errors=True,
    memory=memory,
    max_iterations=5,
    early_stopping_method="generate",
    agent_kwargs={
        "system_message": """You are an intelligent AI assistant with access to 3 tools.

AVAILABLE TOOLS:
1. Calculator → solve math problems and arithmetic
2. Wikipedia Search → search for information and facts about topics
3. Generate Word Document → create Word (.docx) documents

DECISION RULES - WHEN TO USE EACH TOOL:
- Math/Numbers: "What is 2+2?", "Calculate...", "Solve..." → use Calculator
- Information/Facts: "Who is...", "What is...", "Tell me about...", "Search..." → use Wikipedia Search  
- Documents/Writing: "Create document", "Generate word", "Write document", "Make a doc", "Create a file" → use Generate Word Document

IMPORTANT:
- If user says "generate word" or similar → ALWAYS use Generate Word Document tool
- Always use a tool when appropriate
- Provide a direct answer after using tools
- Be concise and clear
- No verbose explanations needed
"""
    }
)

print("✓ Agent initialized with:")
print("  • ConversationBufferMemory (tracks chat history)")
print("  • 3 tools: Calculator, Wikipedia Search, Generate Word Document")
print("  • CHAT_CONVERSATIONAL_REACT_DESCRIPTION agent type")
print("  • Max iterations: 5")


✓ Agent initialized with:
  • ConversationBufferMemory (tracks chat history)
  • 3 tools: Calculator, Wikipedia Search, Generate Word Document
  • CHAT_CONVERSATIONAL_REACT_DESCRIPTION agent type
  • Max iterations: 5


C:\Users\jeric\AppData\Local\Temp\ipykernel_25740\1963731471.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(
C:\Users\jeric\AppData\Local\Temp\ipykernel_25740\1963731471.py:11: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [6]:
# TEST SYSTEM WITH 3 TOOLS

print("\nTESTING AGENT WITH GEMMA 3 4B\n")

test_queries = [
    "What is 15 * 8?",
    "Who is Steve Jobs?",
    "Create a document about artificial intelligence"
]

for i, q in enumerate(test_queries, 1):
    print(f"Test {i}: {q}")
    try:
        response = agent.invoke({"input": q})
        answer = response.get('output', 'No response')
        display_text = answer[:120] + "..." if len(answer) > 120 else answer
        print(f"Response: {display_text}\n")
    except Exception as e:
        print(f"Error: {str(e)[:80]}\n")

print("✓ System ready for interactive chat!")


TESTING AGENT WITH GEMMA 3 4B

Test 1: What is 15 * 8?


> Entering new AgentExecutor chain...
```json
{
    "action": "Calculator",
    "action_input": "15 * 8"
}
```
Observation: 120
Thought:```json
{
    "action": "Final Answer",
    "action_input": "120"
}
```

> Finished chain.
Response: 120

Test 2: Who is Steve Jobs?


> Entering new AgentExecutor chain...
```json
{
    "action": "Wikipedia Search",
    "action_input": "Steve Jobs"
}
```
Observation: Error: Page id "steve jons" does not match any pages. Try another id!
Thought:```json
{
    "action": "Final Answer",
    "action_input": "There was an error searching for information on 'Steve Jons'. The search returned no results."
}
```

> Finished chain.
Response: There was an error searching for information on 'Steve Jons'. The search returned no results.

Test 3: Create a document about artificial intelligence


> Entering new AgentExecutor chain...
```json
{
    "action": "Generate Word Document",
    "action_input": ""
}
``

In [ ]:
# INTERACTIVE CHAT INTERFACE

print("\n" + "="*80)
print("GEMMA AI ASSISTANT - Interactive Chat")
print("="*80)
print("\nAvailable Tools:")
print("  • Calculator - Solve math problems (addition, subtraction, multiplication, division)")
print("  • Wikipedia Search - Get information on any topic (summaries and facts)")
print("  • Document Generator - Create Word documents (.docx files)")
print("\nType 'exit' or 'quit' to end the session\n")
print("="*80 + "\n")

conversation_count = 0

while True:
    try:
        query = input("You: ").strip()
        
        if not query:
            continue
        
        if query.lower() in ['exit', 'quit', 'q']:
            print("\n" + "="*80)
            print("✓ Chat ended. Thank you for using the GEMMA AI Assistant. Goodbye!")
            print("="*80 + "\n")
            break
        
        conversation_count += 1
        print("\nGEMMA: Processing your request...\n")
        
        try:
            response = agent.invoke({"input": query})
            ai_response = response.get('output', 'No response generated')
            
            print("="*80)
            print(f"GEMMA: {ai_response}")
            print("="*80 + "\n")
            
        except Exception as e:
            print("="*80)
            print(f"GEMMA: ❌ Error processing your request: {str(e)}")
            print("="*80 + "\n")
    
    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("✓ Chat interrupted. Goodbye!")
        print("="*80 + "\n")
        break
    except Exception as e:
        print(f"System Error: {str(e)}")
        continue


GEMMA AI ASSISTANT - Interactive Chat

Available Tools:
  • Calculator - Solve math problems (addition, subtraction, multiplication, division)
  • Wikipedia Search - Get information on any topic (summaries and facts)
  • Document Generator - Create Word documents (.docx files)

Type 'exit' or 'quit' to end the session



GEMMA: Processing your request...



> Entering new AgentExecutor chain...
```json
{
    "action": "Calculator",
    "action_input": "10 + 2"
}
```
Observation: 12
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The answer to 10 + 2 is 12."
}
```

> Finished chain.
GEMMA: The answer to 10 + 2 is 12.


GEMMA: Processing your request...



> Entering new AgentExecutor chain...
```json
{
    "action": "Wikipedia Search",
    "action_input": "Big Bang Theory"
}
```
Observation: Error: Page id "big bank theory" does not match any pages. Try another id!
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The Big Bang Theory is the preva

In [ ]:
# DISPLAY ALL GENERATED DOCUMENTS

from pathlib import Path

print("\n" + "="*80)
print("GENERATED DOCUMENTS SUMMARY")
print("="*80 + "\n")

doc_dir = Path("generated_docs")

if doc_dir.exists():
    files = sorted(list(doc_dir.glob("*.docx")), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if files:
        print(f"✓ Total Documents Created: {len(files)}\n")
        print(f"{'#':<4} {'Filename':<40} {'Size':<12} {'Created':<20}")
        print("-" * 80)
        
        for i, f in enumerate(files, 1):
            size_kb = f.stat().st_size / 1024
            mod_time = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
            filename = f.name[:37] + "..." if len(f.name) > 37 else f.name
            
            print(f"{i:<4} {filename:<40} {size_kb:>10.1f} KB  {mod_time:<20}")
        
        print("\n" + "="*80)
        print(f"All documents are saved in: generated_docs/")
        print("="*80 + "\n")
    else:
        print("✓ No documents generated yet.")
        print("\nDocuments will be created when you use the Document Generator tool.")
        print("="*80 + "\n")
else:
    print("✓ Documents folder not created yet.")
    print("\nIt will be created when you generate your first document.")
    print("="*80 + "\n")

In [ ]:
# SYSTEM READY - FINAL STATUS

print("\n" + "="*80)
print("✓ GEMMA AI SYSTEM INITIALIZED AND READY")
print("="*80)
print("\nConfiguration Summary:")
print("  • LLM Model: Gemma 3 4B (google/gemma-3-4b)")
print("  • API Server: http://127.0.0.1:1234/v1")
print("  • Agent Type: CHAT_CONVERSATIONAL_REACT_DESCRIPTION")
print("  • Memory: ConversationBufferMemory (tracks chat history)")
print("  • Max Iterations: 5 per request")
print("\nAvailable Tools (3 Total):")
print("  1. Calculator - Solve arithmetic expressions")
print("  2. Wikipedia Extractor - Search and summarize Wikipedia articles")
print("  3. Word Document Generator - Create formatted .docx files")
print("\nStorage Location:")
print("  • Generated Documents: ./generated_docs/")
print("\nTo start chatting, run the 'INTERACTIVE CHAT INTERFACE' cell above.")
print("="*80 + "\n")